In [1]:
# Importazione pacchetti
import pandas as pd
import datetime as dt

In [2]:
# Importazione dataset
pesature = pd.read_csv(r'Project\03. Variabilità di Peso nel Confezionamento Caffè\pesature.csv')

# 1. DESCRIZIONE DATASET

In [3]:
pesature.head()

,date,shift,batch_id,operator_id,weight_grams
0,2026-06-06,Notte,B1017,OP-01,251.08
1,2026-06-03,Pomeriggio,B1007,OP-02,250.41
2,2026-06-05,Mattina,B1012,OP-04,251.87
3,2026-06-13,Pomeriggio,B1037,OP-02,248.76
4,2026-06-05,Notte,B1014,OP-06,0.00


In [4]:
pesature.info()

<class 'pandas.DataFrame'>
RangeIndex: 424 entries, 0 to 423
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   date          424 non-null    str    
 1   shift         424 non-null    str    
 2   batch_id      424 non-null    str    
 3   operator_id   424 non-null    str    
 4   weight_grams  417 non-null    float64
dtypes: float64(1), str(4)
memory usage: 28.0 KB


In [5]:
pesature.describe()

,weight_grams
count,417.000000
mean,254.570624
std,111.307119
min,0.000000
25%,248.750000
50%,249.910000
75%,250.800000
max,2508.000000


In [6]:
pesature.shape

(424, 5)

# 2. INCONSISTENZE CATEGORIA

In [7]:
# Valori unici colonna shift
pesature['shift'].unique()

<ArrowStringArray>
['Notte', 'Pomeriggio', 'Mattina', 'mattina', 'MATTINA ', 'pomeriggio',
 'notte']
Length: 7, dtype: str

In [8]:
# Ripristino valori corretti
pesature['shift'] = pesature['shift'].str.strip()
pesature['shift'] = pesature['shift'].str.title()

In [9]:
# Controllo valori corretti in shift
pesature['shift'].unique()

<ArrowStringArray>
['Notte', 'Pomeriggio', 'Mattina']
Length: 3, dtype: str

# 3. MISSING VALUES

In [10]:
# Valori nulli per colonna
pesature.isnull().sum()

date            0
shift           0
batch_id        0
operator_id     0
weight_grams    7
dtype: int64

In [11]:
# % valori nulli per colonna
(pesature.isnull().mean()*100).round(1)

date            0.0
shift           0.0
batch_id        0.0
operator_id     0.0
weight_grams    1.7
dtype: float64

In [12]:
# Righe con valori nulli
pesature_null = pesature[pesature.isnull().any(axis =1)]
pesature_null

,date,shift,batch_id,operator_id,weight_grams
85,2026-06-10,Notte,B1029,OP-06,NaN
148,2026-06-01,Pomeriggio,B1001,OP-09,NaN
187,2026-06-06,Pomeriggio,B1016,OP-02,NaN
198,2026-06-09,Pomeriggio,B1025,OP-02,NaN
262,2026-06-08,Mattina,B1021,OP-04,NaN
351,2026-06-13,Pomeriggio,B1037,OP-02,NaN
385,2026-06-11,Pomeriggio,B1031,OP-09,NaN


In [13]:
null_op = pesature_null.groupby(['shift', 'operator_id'])['weight_grams'].agg(
    mancanti = 'size'
)
null_op['%_null'] = (null_op['mancanti']/null_op['mancanti'].sum()*100).round(2)

null_op

mancanti  %_null
shift      operator_id                  
Mattina    OP-04               1   14.29
Notte      OP-06               1   14.29
Pomeriggio OP-02               3   42.86
           OP-09               2   28.57

L'indagine dei valori nulli ha mostrato una maggiore frequenza di questi nel turno del pomeriggio ed assegnati agli operatori OP-02 (42,86%) e OP-09(28,57%). Raccomando di investigare questa tendenza per attestare la causa. 

In [14]:
# Mediana delle pesate per turno
mediana = pesature.groupby('shift')['weight_grams'].transform('median')
mediana
# Sostituzione dei valori nulli con mediana dei dati
pesature['weight_grams'] = pesature['weight_grams'].fillna(mediana)

In [15]:
pesature['weight_grams'].isnull().sum()

np.int64(0)

# 4. OUTLIERS

Essendo che le pesate sono comprese tra 245 e 255, sono rpesenti outliers. Questi li posso identificare ma la sostituzione dei valori deve essere fatta su numeri estremi, non su valori fuori dal range ma realistici.

In [16]:
# Verifico la presenza di outliers all'interno del dataset
# Quartile 1 
Q1 = pesature['weight_grams'].quantile(0.25)

# Quartile 3
Q3 = pesature['weight_grams'].quantile(0.75)

# Range Interquartile
IQR = Q3-Q1

# Limiti
limite_inferiore = Q1-(IQR*1.5)
limite_superiore = Q3+(IQR*1.5)

outliers = pesature[(pesature['weight_grams']<limite_inferiore) | (pesature['weight_grams']>limite_superiore)]
outliers

,date,shift,batch_id,operator_id,weight_grams
4,2026-06-05,Notte,B1014,OP-06,0.00
27,2026-06-13,Mattina,B1036,OP-04,261.40
92,2026-06-14,Notte,B1041,OP-06,244.91
129,2026-06-08,Notte,B1023,OP-06,243.12
136,2026-06-03,Notte,B1008,OP-06,245.57
137,2026-06-11,Notte,B1032,OP-06,2508.00
151,2026-06-09,Notte,B1026,OP-01,244.94
200,2026-06-14,Notte,B1041,OP-06,245.31
220,2026-06-10,Notte,B1029,OP-01,243.77
292,2026-06-06,Notte,B1017,OP-06,245.05


Dalla vista della tabella emergono diversi valori come outliers, proseguo con la ricerca di valori estremi o assurdi impostando il limite inferiore a 200 e quello superiore a 300. Assegnati considerando che le specifiche del cliente non sono paragonabili a questi valori e assumendo che l'impianto, con un range di pesate così ristretto, non possa sbagliare una pesata con uno scostamento dal valore medio così alto.

In [17]:
# Assegno valore minimo
minimo = 200

# Assegno valore massimo
massimo = 300

# Estraggo valori che si trovano sotto il minimo e sopra il massimo
pesature[(pesature['weight_grams']< minimo) | (pesature['weight_grams'] > massimo)]

,date,shift,batch_id,operator_id,weight_grams
4,2026-06-05,Notte,B1014,OP-06,0.0
137,2026-06-11,Notte,B1032,OP-06,2508.0


Dei due valori emersi come errori abbiamo:
- 0.0: verrà sostituito con la mediana delle pesature del turno in cui si localizza il valore (Notte)
- 2508.8: Lo associo ad un errore di battitura e verrà diviso per 10 per ripristinarlo ai range prestabiliti

In [18]:


# Sostituisco il valore 0.0 presente nel dataset con la mediana
pesature.loc[pesature['weight_grams'] == 0, 'weight_grams'] = mediana

# Modifico l'errore di battitura dividendo per 10 il dato 2508.0
pesature.loc[pesature['weight_grams'] == 2508.0, 'weight_grams'] = 2508.0/10

In [19]:
# Verifico la presenza di altri valori estremi dopo la modifica
pesature[(pesature['weight_grams']< minimo) | (pesature['weight_grams'] > massimo)].sum()

date               
shift              
batch_id           
operator_id        
weight_grams    0.0
dtype: object

# 5. DUPLICATI

In [20]:
# Conteggio dei valori duplicati
pesature.duplicated().sum()

np.int64(4)

In [21]:
# Eliminazione duplicati
pesature_singole = pesature.drop_duplicates()

In [22]:
pesature_singole.duplicated().sum

<bound method Series.sum of 0      False
1      False
2      False
3      False
4      False
       ...  
419    False
420    False
421    False
422    False
423    False
Length: 420, dtype: bool>

# 6. CORREZIONE FORMATI DATASET

In [23]:
def data_quality_report(df):
    report = pd.DataFrame({
        'tipo': df.dtypes,
        'missing': df.isnull().sum(),
        'missing_%': (df.isnull().mean() * 100).round(2),
        'valori_unici': df.nunique()
    })
    print(report)
    print(f"\nRighe totali: {len(df)}")
    print(f"Duplicati: {df.duplicated().sum()}")

data_quality_report(pesature_singole)


                 tipo  missing  missing_%  valori_unici
date              str        0        0.0            17
shift             str        0        0.0             3
batch_id          str        0        0.0            42
operator_id       str        0        0.0             8
weight_grams  float64        0        0.0           304

Righe totali: 420
Duplicati: 0


In [24]:
date_originale = pesature_singole['date'].copy()   # backup prima di toccare la colonna

# 1. Primo tentativo
pesature_singole['date'] = pd.to_datetime(date_originale, format='%Y-%m-%d', errors='coerce')

# 2. Righe non riconosciute con il primo formato
mask_fail = pesature_singole['date'].isna()

# 3. Riprova SOLO su quelle righe, con il formato alternativo, leggendo dall'originale
pesature_singole.loc[mask_fail, 'date'] = pd.to_datetime(
    date_originale[mask_fail], format='%d/%m/%Y', errors='coerce'
)

# 4. Verifica: quanti NaT restano DAVVERO (dati non recuperabili con nessuno dei due formati)?
pesature_singole['date'].isna().sum()

np.int64(0)

In [25]:
def data_quality_report(df):
    report = pd.DataFrame({
        'tipo': df.dtypes,
        'missing': df.isnull().sum(),
        'missing_%': (df.isnull().mean() * 100).round(2),
        'valori_unici': df.nunique()
    })
    print(report)
    print(f"\nRighe totali: {len(df)}")
    print(f"Duplicati: {df.duplicated().sum()}")

data_quality_report(pesature_singole)

                        tipo  missing  missing_%  valori_unici
date          datetime64[us]        0        0.0            14
shift                    str        0        0.0             3
batch_id                 str        0        0.0            42
operator_id              str        0        0.0             8
weight_grams         float64        0        0.0           304

Righe totali: 420
Duplicati: 0


Pulito il datase e sistemato i formati delle colonne esporto in formato CSV per l'analisi dati

In [26]:
pd.DataFrame.to_csv(pesature_singole, r'C:\Users\nicol\DataAnalysis\Project\03. Variabilità di Peso nel Confezionamento Caffè\Dataset\Clean\pesature_clean.csv', index=False)